# Tasks 1.2–1.3 — Simulador, revisión y pruebas


## Task 1.2 — Anexo A (prompt)
Pegar aquí **el prompt exacto** y **el código tal como lo recibió** de la IA que usted use, sin modificar (requisito del enunciado). Prompt sugerido:

> Escribe en Python un simulador de eventos discretos con lista de eventos para dos variantes: M/M/c (cola infinita FIFO) y M/M/c/c (el cliente se pierde si no hay servidor libre). Entradas: lambda, mu, c, horizonte, calentamiento y semilla. Reporta L, Lq, Wq, utilización, probabilidad de esperar y probabilidad de bloqueo. Si ejecutas código, indica qué pruebas hiciste y si los resultados coinciden con la teoría.

Pegar también literalmente las afirmaciones del agente sobre su corrección (se usan en 1.4c).

> El código de abajo es el simulador **corregido** (Task 1.3c). Las pruebas del resto del notebook están escritas para correrse contra *cualquier* simulador con la misma firma, así que sirven para auditar el original.

In [1]:
# --- Parámetros del grupo ---
import math
g = 3                                   # <-- número de grupo (cambiar aquí)
mu = 0.25
lam = 3.0 + 0.2 * ((g - 1) % 5)
c = 17 + ((g - 1) % 3)
a = lam / mu
rho = a / c
k_off = math.ceil(c - a) + 3
Tb = 7
print(f"Grupo {g}: lambda={lam:.1f}/día, c={c}, a={a:.2f}, rho={rho:.4f}, k_off={k_off}, Tb={Tb} días")

import numpy as np

def erlang_b(c, a):
    """Recurrencia: B(0)=1;  B(k) = a B(k-1) / (k + a B(k-1))"""
    B = 1.0
    for k in range(1, c + 1):
        B = a * B / (k + a * B)
    return B

def erlang_c_direct(c, a):
    """C(c,a) = [a^c/c! /(1-rho)] / [sum_{k<c} a^k/k! + a^c/c! /(1-rho)]  (estable vía logs)"""
    from math import lgamma, log, exp
    rho = a / c
    terms = [exp(k*log(a) - lgamma(k+1)) for k in range(c)]
    top = exp(c*log(a) - lgamma(c+1)) / (1 - rho)
    return top / (sum(terms) + top)

def theory_wait(lam, mu, c):
    a = lam / mu; rho = a / c
    assert rho < 1
    B = erlang_b(c, a)
    C = B / (1 - rho * (1 - B))             # Erlang C a partir de Erlang B
    Lq = C * rho / (1 - rho)
    Wq = Lq / lam                            # = C/(c mu - lam)
    W = Wq + 1 / mu
    L = Lq + a
    return dict(a=a, rho=rho, C=C, Lq=Lq, Wq_h=Wq*24, W_h=W*24, Wq=Wq, W=W, L=L, util=rho, Pwait=C)

def theory_loss(lam, mu, c):
    a = lam / mu
    B = erlang_b(c, a)
    lam_eff = lam * (1 - B)
    L = lam_eff / mu                         # = a (1-B)
    return dict(B=B, lam_eff=lam_eff, L=L, util=L / c, W=1/mu, Pblock=B)


Grupo 3: lambda=3.4/día, c=19, a=13.60, rho=0.7158, k_off=9, Tb=7 días


## Simulador corregido

In [2]:
import heapq, itertools
from collections import deque
import numpy as np
from scipy import stats

def simulate(lam, mu, c, T, warmup=0.0, seed=None, wait=True,
             inter_fn=None, serv_fn=None, sample_dt=None, _bugs=()):
    """Simulación de eventos discretos con lista de eventos (heap). Tiempo en días.
    wait=True  -> M/M/c   (cola infinita FIFO)
    wait=False -> M/M/c/c (si no hay cama, el paciente se pierde)
    inter_fn / serv_fn: generadores alternativos (para pruebas deterministas).
    _bugs: SOLO para pruebas de mutación (inyectan fallas conocidas)."""
    rng = np.random.default_rng(seed)
    ia = inter_fn or (lambda: rng.exponential(1.0 / lam))
    sv = serv_fn or (lambda: rng.exponential(1.0 / mu))
    if 'no_warmup' in _bugs:
        warmup = 0.0
    ev, cnt = [], itertools.count()
    heapq.heappush(ev, (ia(), next(cnt), 'A', None))
    heapq.heappush(ev, (T, next(cnt), 'END', None))
    if warmup > 0:
        heapq.heappush(ev, (warmup, next(cnt), 'WARM', None))

    busy, q = 0, deque()                       # q: [t_llegada, contado, t_cabeza]
    last, meas = 0.0, warmup <= 0
    aL = aLq = aB = 0.0                        # integrales ponderadas por duración
    naive_sum = naive_n = 0
    n_arr = n_blk = n_delayed = 0
    waits, soj = [], []
    S_n, S_q, next_s = [], [], 0.0

    def acc(t):
        nonlocal aL, aLq, aB
        dt = t - last
        aL += (busy + len(q)) * dt; aLq += len(q) * dt; aB += busy * dt

    while ev:
        t, _, kind, data = heapq.heappop(ev)
        if sample_dt:
            while next_s <= min(t, T):
                S_n.append(busy + len(q)); S_q.append(len(q)); next_s += sample_dt
        if kind == 'END':
            if meas and 'no_final_segment' not in _bugs:
                acc(T)                         # cierra el último tramo hasta T
            break
        if meas:
            acc(t)
        last = t
        if kind == 'WARM':
            meas = True
        elif kind == 'A':
            heapq.heappush(ev, (t + ia(), next(cnt), 'A', None))
            counted = meas                     # la llegada pertenece a la ventana de medición
            n_arr += counted
            if busy < c:
                busy += 1
                if counted: waits.append(0.0)
                heapq.heappush(ev, (t + sv(), next(cnt), 'D', (t, counted)))
            elif wait:
                if counted: n_delayed += 1
                q.append([t, counted, t])
            else:
                n_blk += counted
        elif kind == 'D':
            arr, counted = data
            busy -= 1
            if counted: soj.append(t - arr)    # tiempo en sistema (llegada -> salida)
            if q:
                a_t, cn, head = q.popleft()
                busy += 1
                w = (t - head) if 'wait_from_head' in _bugs else (t - a_t)   # espera desde LLEGADA
                if cn: waits.append(w)
                heapq.heappush(ev, (t + sv(), next(cnt), 'D', (a_t, cn)))
                if q: q[0][2] = t
        if meas:
            naive_sum += busy + len(q); naive_n += 1

    span = T - warmup
    L = aL / span
    if 'naive_L' in _bugs:
        L = naive_sum / max(naive_n, 1)
    admitted = n_arr - n_blk
    return dict(L=L, Lq=aLq / span, util=aB / span / c,
                Pwait=(n_delayed / n_arr) if (wait and n_arr) else (0.0 if wait else np.nan),
                Pblock=(n_blk / n_arr) if (n_arr and not wait) else (np.nan if wait else 0.0),
                Wq=float(np.mean(waits)) if waits else np.nan,
                W=float(np.mean(soj)) if soj else np.nan,
                lam_eff=admitted / span, n_arr=n_arr, n_blk=n_blk, span=span,
                waits=np.array(waits), soj=np.array(soj),
                S_n=np.array(S_n), S_q=np.array(S_q))

def run_reps(R, lam, mu, c, T, warmup, wait=True, master_seed=2026, _bugs=(), **kw):
    """R réplicas con flujos de números aleatorios independientes (SeedSequence.spawn)."""
    seeds = np.random.SeedSequence(master_seed).spawn(R)
    if 'same_seed' in _bugs:
        seeds = [master_seed] * R
    return [simulate(lam, mu, c, T, warmup, seed=s, wait=wait, _bugs=_bugs, **kw) for s in seeds]

def ci95(x):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    m = x.mean(); h = stats.t.ppf(0.975, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x))
    return m, h


## Pruebas con respuesta conocida (diseñadas a mano)
Casos deterministas con `inter_fn`/`serv_fn` fijos. Todos con **1 cama**.

| Caso | Config | Resultado a mano |
|---|---|---|
| D1 (i) | M/M/1/1, inter=1, serv=1.5, T=1000.5, sin calentamiento | llegadas en t=1..1000; admitidas las impares → B=0.5; ocupada 500×1.5 = 750 d → L=750/1000.5 |
| D2 (iii) | igual, calentamiento 500.25, T=1000.2 | ventana de 499.95 d; ocupación = 0.25 + 249×1.5 + 1.2 = 374.95; llegadas 500, bloqueadas 250 |
| D3 (vi) | igual a D1, T=1000.2 | el paciente de t=999 queda dentro; L=(499×1.5+1.2)/1000.2 |
| D4 (vii) | M/M/1, inter=1, serv=2, T=100.5 | la espera del cliente k es k−1; empiezan servicio k≤50 → Wq=24.5 |

In [3]:
def const(x):
    return lambda: x

def tests(sim, rep, name=""):
    """sim(...) y rep(...) con la firma de arriba. Devuelve dict test->bool."""
    res = {}
    # (i) promedio ponderado por duración
    r = sim(0, 0, 1, 1000.5, wait=False, inter_fn=const(1.0), serv_fn=const(1.5), warmup=0.0)
    res['i  L ponderado (D1)'] = abs(r['L'] - 750/1000.5) < 1e-9 and abs(r['Pblock'] - 0.5) < 1e-12
    # (iii) calentamiento descarta lo previo
    r = sim(0, 0, 1, 1000.2, wait=False, inter_fn=const(1.0), serv_fn=const(1.5), warmup=500.25)
    res['iii calentamiento (D2)'] = abs(r['L'] - 374.95/499.95) < 1e-9 and r['n_arr'] == 500 and r['n_blk'] == 250
    # (iv) semillas independientes entre réplicas
    reps = rep(10, 3.4, 0.25, 19, 60.0, 10.0, wait=True)
    Ls = np.array([x['L'] for x in reps])
    res['iv semillas independientes'] = len(set(np.round(Ls, 12))) == 10 and Ls.std() > 0
    # (v) tasa efectiva: M/M/2/2 con a=2 (B=0.4): Little con lam_ef cierra; con lam no
    reps = rep(30, 2.0, 1.0, 2, 3000.0, 50.0, wait=False)
    e_eff = np.mean([abs(x['L'] - x['lam_eff'] * x['W']) / x['L'] for x in reps])
    e_lam = np.mean([abs(x['L'] - 2.0 * x['W']) / x['L'] for x in reps])
    res['v  tasa efectiva (Little)'] = e_eff < 0.01 and e_lam > 0.3
    # (vi) entidades dentro al cerrar el horizonte
    r = sim(0, 0, 1, 1000.2, wait=False, inter_fn=const(1.0), serv_fn=const(1.5), warmup=0.0)
    res['vi leftover consistente (D3)'] = abs(r['L'] - (499*1.5 + 1.2)/1000.2) < 1e-9
    # (vii) espera desde la llegada
    r = sim(0, 0, 1, 100.5, wait=True, inter_fn=const(1.0), serv_fn=const(2.0), warmup=0.0)
    res['vii espera desde llegada (D4)'] = abs(r['Wq'] - 24.5) < 1e-9
    return res

R = tests(simulate, run_reps)
for k, v in R.items(): print(("OK   " if v else "FALLA"), k)
assert all(R.values())

OK    i  L ponderado (D1)
OK    iii calentamiento (D2)
OK    iv semillas independientes
OK    v  tasa efectiva (Little)
OK    vi leftover consistente (D3)
OK    vii espera desde llegada (D4)


## Pruebas de mutación
Se inyectan fallas conocidas (las del listado) para comprobar que **cada prueba detecta su falla**; es la evidencia de que las pruebas sirven para auditar el código original de la IA.

In [4]:
import pandas as pd
mutants = {'(ninguno: corregido)': (), 'i  naive_L': ('naive_L',), 'iii no_warmup': ('no_warmup',),
           'iv  same_seed': ('same_seed',), 'vi  no_final_segment': ('no_final_segment',),
           'vii wait_from_head': ('wait_from_head',)}
rows = {}
for nm, b in mutants.items():
    sim_m = lambda *a, _b=b, **k: simulate(*a, _bugs=_b, **k)
    rep_m = lambda *a, _b=b, **k: run_reps(*a, _bugs=_b, **k)
    rows[nm] = tests(sim_m, rep_m)
pd.DataFrame(rows).T.replace({True: 'pasa', False: 'FALLA'})

,i L ponderado (D1),iii calentamiento (D2),iv semillas independientes,v tasa efectiva (Little),vi leftover consistente (D3),vii espera desde llegada (D4)
(ninguno: corregido),pasa,pasa,pasa,pasa,pasa,pasa
i naive_L,FALLA,FALLA,pasa,FALLA,FALLA,pasa
iii no_warmup,pasa,FALLA,pasa,pasa,pasa,pasa
iv same_seed,pasa,pasa,FALLA,pasa,pasa,pasa
vi no_final_segment,FALLA,FALLA,pasa,pasa,FALLA,pasa
vii wait_from_head,pasa,pasa,pasa,pasa,pasa,FALLA


**(v) y (ii)** son errores de *análisis*, no de la dinámica: (v) lo detecta la prueba de Little de arriba (error ≈ 0 con $\lambda_{ef}$ vs. ≈ 40 %+ con $\lambda$ cuando $B\approx0.4$); (ii) se detecta comparando el IC por entidad contra el IC por réplica (ver Notebook 3, celda de unidad de muestreo).

## Task 1.3b — Tabla de revisión
La columna **«Cumple (original)»** depende del código que le entregue *su* IA: córrale `tests(...)` (adaptando la firma) y complétela. Las demás columnas ya están definidas.

| # | Punto | Cumple (original) | Evidencia a buscar | Supuesto que viola | Prueba concreta |
|---|---|---|---|---|---|
| i | Promedio ponderado por duración | *llenar* | `acc()`: `area += n*(t-last)` y `L=area/span`; falla si promedia `n` en cada evento | PASTA no aplica a instantes de evento: llegadas/salidas sesgan el muestreo | D1: caso 1 cama determinista, L=750/1000.5 (el promedio por evento da 2/3) |
| ii | Unidad de muestreo = réplica | *llenar* | IC calculado con `std` de las réplicas, no de las `waits` | Independencia de observaciones (las esperas consecutivas están autocorrelacionadas) | Comparar semi-ancho IC por entidad vs réplica (NB3) |
| iii | Calentamiento descartado y justificado | *llenar* | evento `WARM` y `arr>=warmup` | Estado estacionario desde t=0 (arranca vacío → L sesgado a la baja) | D2 + gráfico Welch (NB3) |
| iv | Semillas independientes | *llenar* | `SeedSequence.spawn` | Independencia entre réplicas | 10 réplicas: L distintos, std>0 (con misma semilla std=0) |
| v | Tasa efectiva en sin espera | *llenar* | `lam_eff = admitidas/span` en Little y en utilización | Todos los llegados son atendidos | M/M/2/2, a=2 (B=0.4): Little con $\lambda_{ef}$ cierra |
| vi | Entidades dentro al cierre | *llenar* | integral cerrada en `T` (`acc(T)`) y censura documentada en W/Wq | Sistema vacío al final | D3: L=(499·1.5+1.2)/1000.2 |
| vii | Espera desde la llegada | *llenar* | `t - a_t` con `a_t` guardado al llegar | Reloj de espera arranca en la llegada, no al llegar a la cabeza | D4: Wq=24.5 |

### Task 1.3c — Evaluación del código original
Redactar tras correr las pruebas sobre su código. Estructura sugerida: (1) cuántos de los 7 puntos pasaban; (2) cuáles errores solo aparecen con una prueba de respuesta conocida (típicamente i, vi, vii: el simulador «parece» razonable y los promedios salen cerca de la teoría); (3) nivel de confianza como punto de partida: útil como esqueleto, no firmable sin las pruebas D1–D4.